# Predviđanje Performansi NFL Igrača
## NFL Player Performance Prediction using Machine Learning

**Autori:** Milan Jovkić R2 10/2025, Uroš Petrašković R2 9/2025

---

### 1. Definicija problema

Potrebno je razviti model za predviđanje budućih performansi NFL igrača. Korišćenjem statističkih podataka o igračima iz prethodnih sezona, model treba da proceni performanse igrača u budućnosti.

### Ciljne promenljive po pozicijama:
- **Quarterback (QB):** Passing Yards
- **Running Back (RB):** Rushing Yards  
- **Wide Receiver (WR):** Receiving Yards
- **Tight End (TE):** Receiving Yards

### Metodologija:
Primena različitih regresionih algoritama:
- Linear Regression (baseline)
- Ridge, Lasso, ElasticNet (regularizovane regresije)
- K-Neighbors Regressor
- **Random Forest Regression** (primarni model)
- Multi-Output Regression (proširenje)

### Metrike evaluacije:
- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- R² (Coefficient of Determination)

## 1. Import Required Libraries

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Machine Learning models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

# Model selection and evaluation
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Model persistence
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print("All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Data Collection and Loading

Učitavanje podataka za svaku poziciju:
- **QB (Quarterback):** `all_qb_stats_full.csv` - prikupljeno sa Pro Football Reference
- **RB (Running Back):** `all_rushing_receiving.csv` + `all_advanced_rushing_receiving.csv`
- **TE (Tight End):** `all_te_receiving_rushing.csv` + `all_te_advanced_receiving_rushing.csv`
- **WR (Wide Receiver):** HuggingFace dataset (2015-2025)

In [ ]:
# Define data directories
DATA_DIR = 'data/processed'
RAW_DIR = 'data/raw'

# Load Quarterback (QB) data
print("=" * 60)
print("Loading Quarterback (QB) Data")
print("=" * 60)
qb_df = pd.read_csv(os.path.join(DATA_DIR, 'all_qb_stats_full.csv'))
print(f"Shape: {qb_df.shape}")
print(f"Seasons: {sorted(qb_df['Season'].unique())}")
print(f"Number of unique players: {qb_df['Player'].nunique()}")
print(f"\nFirst few rows:")
qb_df.head()

In [ ]:
# Load Running Back (RB) data
print("=" * 60)
print("Loading Running Back (RB) Data")
print("=" * 60)

rb_basic = pd.read_csv(os.path.join(DATA_DIR, 'all_rushing_receiving.csv'))
rb_advanced = pd.read_csv(os.path.join(DATA_DIR, 'all_advanced_rushing_receiving.csv'))

# Merge basic and advanced stats
merge_cols = ['Season', 'Player', 'PlayerID']
rb_df = rb_basic.merge(rb_advanced, on=merge_cols, how='left', suffixes=('', '_adv'))

print(f"Basic stats shape: {rb_basic.shape}")
print(f"Advanced stats shape: {rb_advanced.shape}")
print(f"Merged shape: {rb_df.shape}")
print(f"Seasons: {sorted(rb_df['Season'].unique())}")
print(f"Number of unique players: {rb_df['Player'].nunique()}")
rb_df.head()

In [ ]:
# Load Tight End (TE) data
print("=" * 60)
print("Loading Tight End (TE) Data")
print("=" * 60)

te_basic = pd.read_csv(os.path.join(DATA_DIR, 'all_te_receiving_rushing.csv'))
try:
    te_advanced = pd.read_csv(os.path.join(DATA_DIR, 'all_te_advanced_receiving_rushing.csv'))
    te_df = te_basic.merge(te_advanced, on=['Season', 'Player', 'PlayerID'], how='left', suffixes=('', '_adv'))
except FileNotFoundError:
    te_df = te_basic

print(f"Shape: {te_df.shape}")
print(f"Seasons: {sorted(te_df['Season'].unique())}")
print(f"Number of unique players: {te_df['Player'].nunique()}")
te_df.head()

In [ ]:
# Load Wide Receiver (WR) data from HuggingFace dataset
print("=" * 60)
print("Loading Wide Receiver (WR) Data")
print("=" * 60)

wr_files = []
wr_base_dir = os.path.join(RAW_DIR, 'wr')

for year in range(2015, 2026):
    year_file = os.path.join(wr_base_dir, str(year), 'data', f'wr_{year}.csv')
    if os.path.exists(year_file):
        df = pd.read_csv(year_file)
        df['Season'] = year
        wr_files.append(df)
        print(f"  Loaded {year}: {len(df)} records")

if wr_files:
    wr_df = pd.concat(wr_files, ignore_index=True)
    print(f"\nTotal WR records: {len(wr_df)}")
    print(f"Columns: {len(wr_df.columns)}")
    print(f"Unique players: {wr_df['receiver_player_name'].nunique() if 'receiver_player_name' in wr_df.columns else 'N/A'}")
else:
    wr_df = None
    print("No WR data files found")

if wr_df is not None:
    wr_df.head()

## 3. Exploratory Data Analysis (EDA)

Analiza distribucije ciljnih promenljivih i korelacija između feature-a i targeta.

In [ ]:
# QB Data Analysis - Target: Passing Yards (Yds)
print("=" * 60)
print("QB EXPLORATORY DATA ANALYSIS")
print("Target Variable: Passing Yards")
print("=" * 60)

# Filter meaningful records (at least 4 games played)
qb_filtered = qb_df[qb_df['G'] >= 4].copy()
print(f"Records with G >= 4: {len(qb_filtered)} (filtered from {len(qb_df)})")

# Distribution of target variable
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Histogram
axes[0].hist(qb_filtered['Yds'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Passing Yards')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of QB Passing Yards')
axes[0].axvline(qb_filtered['Yds'].mean(), color='red', linestyle='--', label=f'Mean: {qb_filtered["Yds"].mean():.0f}')
axes[0].legend()

# Box plot by season
qb_recent = qb_filtered[qb_filtered['Season'] >= 2015]
sns.boxplot(data=qb_recent, x='Season', y='Yds', ax=axes[1])
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Passing Yards')
axes[1].set_title('Passing Yards by Season')
axes[1].tick_params(axis='x', rotation=45)

# Scatter: Age vs Yards
axes[2].scatter(qb_filtered['Age'], qb_filtered['Yds'], alpha=0.5)
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Passing Yards')
axes[2].set_title('Age vs Passing Yards')

plt.tight_layout()
plt.show()

# Summary statistics
print("\nPassing Yards Summary Statistics:")
print(qb_filtered['Yds'].describe())

In [ ]:
# Correlation analysis for QB
print("Top correlations with Passing Yards:")

# Select numeric columns for correlation
numeric_cols = qb_filtered.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['Season', 'PlayerID']]

# Calculate correlations with target
correlations = qb_filtered[numeric_cols].corr()['Yds'].drop('Yds').sort_values(ascending=False)

# Plot top correlations
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top positive correlations
top_positive = correlations.head(15)
axes[0].barh(range(len(top_positive)), top_positive.values)
axes[0].set_yticks(range(len(top_positive)))
axes[0].set_yticklabels(top_positive.index)
axes[0].invert_yaxis()
axes[0].set_xlabel('Correlation with Passing Yards')
axes[0].set_title('Top 15 Positive Correlations')

# Top negative correlations (potential inverse relationships)
top_negative = correlations.tail(10)
axes[1].barh(range(len(top_negative)), top_negative.values)
axes[1].set_yticks(range(len(top_negative)))
axes[1].set_yticklabels(top_negative.index)
axes[1].invert_yaxis()
axes[1].set_xlabel('Correlation with Passing Yards')
axes[1].set_title('Negative Correlations')

plt.tight_layout()
plt.show()

print("\nTop 10 Correlated Features:")
print(correlations.head(10))

In [ ]:
# RB Data Analysis - Target: Rushing Yards (Yds)
print("=" * 60)
print("RB EXPLORATORY DATA ANALYSIS")
print("Target Variable: Rushing Yards")
print("=" * 60)

rb_filtered = rb_df[rb_df['G'] >= 4].copy()
print(f"Records with G >= 4: {len(rb_filtered)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Histogram of Rushing Yards
axes[0].hist(rb_filtered['Yds'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0].set_xlabel('Rushing Yards')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of RB Rushing Yards')
axes[0].axvline(rb_filtered['Yds'].mean(), color='red', linestyle='--', label=f'Mean: {rb_filtered["Yds"].mean():.0f}')
axes[0].legend()

# Yards per Attempt distribution
axes[1].hist(rb_filtered['Y/A'].dropna(), bins=25, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Yards per Attempt')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Yards per Attempt')

# Attempts vs Yards scatter
axes[2].scatter(rb_filtered['Att'], rb_filtered['Yds'], alpha=0.5, color='green')
axes[2].set_xlabel('Rushing Attempts')
axes[2].set_ylabel('Rushing Yards')
axes[2].set_title('Attempts vs Rushing Yards')

plt.tight_layout()
plt.show()

print("\nRushing Yards Summary Statistics:")
print(rb_filtered['Yds'].describe())

## 4. Data Preprocessing

Priprema podataka za modelovanje:
1. **Rad sa nedostajućim vrednostima** - imputacija medijanom
2. **Detekcija i tretman outliera** - IQR metod
3. **Enkodovanje kategorijskih promenljivih** - LabelEncoder
4. **Normalizacija i skaliranje** - StandardScaler

In [ ]:
def preprocess_data(df, feature_cols, target_col, treat_outliers=True):
    """
    Preprocess data for modeling.
    
    Steps:
    1. Handle missing values (median imputation)
    2. Treat outliers using IQR method (clip)
    3. Scale features using StandardScaler
    """
    df_processed = df.copy()
    
    # 1. Handle missing values
    print("Step 1: Handling missing values...")
    available_features = [c for c in feature_cols if c in df_processed.columns]
    
    for col in available_features + [target_col]:
        if col in df_processed.columns:
            missing_count = df_processed[col].isna().sum()
            if missing_count > 0:
                df_processed[col].fillna(df_processed[col].median(), inplace=True)
    
    # 2. Treat outliers using IQR method
    if treat_outliers:
        print("Step 2: Treating outliers (IQR clipping)...")
        for col in available_features:
            if col in df_processed.columns and df_processed[col].dtype in ['int64', 'float64']:
                Q1 = df_processed[col].quantile(0.25)
                Q3 = df_processed[col].quantile(0.75)
                IQR = Q3 - Q1
                lower = Q1 - 1.5 * IQR
                upper = Q3 + 1.5 * IQR
                df_processed[col] = df_processed[col].clip(lower=lower, upper=upper)
    
    # 3. Extract features and target
    print("Step 3: Extracting features and target...")
    X = df_processed[available_features].copy()
    y = df_processed[target_col].copy()
    
    # 4. Scale features
    print("Step 4: Scaling features with StandardScaler...")
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
    
    print(f"Preprocessing complete. Features: {X_scaled.shape[1]}, Samples: {X_scaled.shape[0]}")
    
    return X_scaled, y, scaler, available_features


def create_time_split(df, season_col='Season', test_seasons=2):
    """Create chronological train-test split based on seasons."""
    unique_seasons = sorted(df[season_col].unique())
    test_cutoff = unique_seasons[-test_seasons]
    
    train_df = df[df[season_col] < test_cutoff].copy()
    test_df = df[df[season_col] >= test_cutoff].copy()
    
    print(f"Training seasons: {unique_seasons[:-test_seasons]}")
    print(f"Testing seasons: {unique_seasons[-test_seasons:]}")
    print(f"Train: {len(train_df)}, Test: {len(test_df)}")
    
    return train_df, test_df

print("Preprocessing functions defined successfully!")

## 5. Feature Engineering

Definisanje relevantnih feature-a za svaku poziciju na osnovu domen znanja i advanced statistika.

In [ ]:
# Define feature sets for each position

# QB Features - for predicting Passing Yards
QB_FEATURES = [
    # Basic statistics
    'Age', 'G', 'GS', 'Cmp', 'Att', 'Cmp%', 'TD', 'TD%', 'Int', 'Int%',
    '1D', 'Succ%', 'Lng', 'Y/A', 'AY/A', 'Y/C', 'Rate', 'Sk', 'Yds_Lost', 'Sk%',
    'NY/A', 'ANY/A', 
    # Rushing stats
    'Rush_Att', 'Rush_Yds', 'Rush_TD', 'Rush_1D', 'Rush_Y/A',
    # Advanced stats
    'pass_air_yds', 'pass_yac', 'pass_drops', 'pass_drop_pct',
    'pass_poor_throws', 'pass_on_target_pct', 'pocket_time',
    'pass_blitzed', 'pass_hurried', 'pass_pressured', 'pass_pressured_pct'
]

# RB Features - for predicting Rushing Yards
RB_FEATURES = [
    'Age', 'G', 'GS', 'Att', 'TD', '1D', 'Succ%', 'Lng', 'Y/A', 'Y/G', 'A/G',
    # Receiving stats
    'Tgt', 'Rec', 'Yds.1', 'Y/R', 'TD.1', '1D.1', 'rec_success', 'R/G',
    'catch_pct', 'Y/Tgt', 'Touch', 'yds_per_touch',
    # Advanced rushing
    'YBC', 'YBC/Att', 'YAC', 'YAC/Att', 'BrkTkl', 'Att/Br'
]

# TE Features - for predicting Receiving Yards
TE_FEATURES = [
    'Age', 'G', 'GS', 'Tgt', 'Rec', 'Y/R', 'TD', '1D', 'rec_success',
    'rec_long', 'R/G', 'Y/G', 'catch_pct', 'Y/Tgt', 'Touch', 'yds_per_touch'
]

print("Feature sets defined:")
print(f"  QB: {len(QB_FEATURES)} features")
print(f"  RB: {len(RB_FEATURES)} features")
print(f"  TE: {len(TE_FEATURES)} features")

## 6. Train-Test Split by Season

Hronološki pristup - korišćenje prethodnih sezona za treniranje i predikcija performansi u narednim sezonama.

In [ ]:
# Prepare QB data for modeling
print("=" * 60)
print("PREPARING QB DATA FOR MODELING")
print("=" * 60)

# Filter to players with meaningful playing time
qb_model_df = qb_df[qb_df['G'] >= 4].copy()

# Split by season
qb_train, qb_test = create_time_split(qb_model_df, 'Season', test_seasons=2)

# Preprocess training data
print("\nPreprocessing training data...")
X_train_qb, y_train_qb, qb_scaler, qb_used_features = preprocess_data(
    qb_train, QB_FEATURES, 'Yds'
)

# Preprocess test data using training statistics
print("\nPreprocessing test data...")
available_qb_features = [c for c in qb_used_features if c in qb_test.columns]

# Handle missing values
qb_test_processed = qb_test.copy()
for col in available_qb_features + ['Yds']:
    if col in qb_test_processed.columns:
        qb_test_processed[col].fillna(qb_train[col].median(), inplace=True)

X_test_qb = qb_test_processed[available_qb_features].copy()
y_test_qb = qb_test_processed['Yds'].copy()

# Scale test features
X_test_qb_scaled = pd.DataFrame(
    qb_scaler.transform(X_test_qb), 
    columns=X_test_qb.columns, 
    index=X_test_qb.index
)

print(f"\nQB Training set: {X_train_qb.shape}")
print(f"QB Test set: {X_test_qb_scaled.shape}")

In [ ]:
# Prepare RB data for modeling
print("=" * 60)
print("PREPARING RB DATA FOR MODELING")
print("=" * 60)

rb_model_df = rb_df[rb_df['G'] >= 4].copy()
rb_train, rb_test = create_time_split(rb_model_df, 'Season', test_seasons=2)

print("\nPreprocessing training data...")
X_train_rb, y_train_rb, rb_scaler, rb_used_features = preprocess_data(
    rb_train, RB_FEATURES, 'Yds'
)

print("\nPreprocessing test data...")
available_rb_features = [c for c in rb_used_features if c in rb_test.columns]

rb_test_processed = rb_test.copy()
for col in available_rb_features + ['Yds']:
    if col in rb_test_processed.columns:
        rb_test_processed[col].fillna(rb_train[col].median(), inplace=True)

X_test_rb = rb_test_processed[available_rb_features].copy()
y_test_rb = rb_test_processed['Yds'].copy()

X_test_rb_scaled = pd.DataFrame(
    rb_scaler.transform(X_test_rb), 
    columns=X_test_rb.columns, 
    index=X_test_rb.index
)

print(f"\nRB Training set: {X_train_rb.shape}")
print(f"RB Test set: {X_test_rb_scaled.shape}")

## 7. Model Training and Evaluation Functions

Definisanje funkcija za treniranje i evaluaciju različitih modela.

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Model"):
    """
    Train and evaluate a regression model.
    
    Returns:
        Dictionary with MAE, RMSE, R² metrics for both train and test sets
    """
    # Train the model
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    metrics = {
        'model': model_name,
        'train_mae': mean_absolute_error(y_train, y_train_pred),
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'train_r2': r2_score(y_train, y_train_pred),
        'test_mae': mean_absolute_error(y_test, y_test_pred),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'test_r2': r2_score(y_test, y_test_pred)
    }
    
    return metrics, y_test_pred


def compare_all_models(X_train, y_train, X_test, y_test, position_name="Position"):
    """
    Compare all regression models on the given dataset.
    
    Models tested:
    - Linear Regression (baseline)
    - Ridge Regression
    - Lasso Regression
    - ElasticNet
    - K-Neighbors Regressor
    - Random Forest Regressor
    - Gradient Boosting Regressor
    """
    print(f"\n{'='*60}")
    print(f"MODEL COMPARISON: {position_name}")
    print(f"{'='*60}")
    
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge (α=1.0)': Ridge(alpha=1.0, random_state=42),
        'Lasso (α=0.1)': Lasso(alpha=0.1, random_state=42, max_iter=10000),
        'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=10000),
        'KNN (k=5)': KNeighborsRegressor(n_neighbors=5, weights='distance'),
        'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
    }
    
    results = []
    predictions = {}
    
    for name, model in models.items():
        print(f"Training {name}...")
        metrics, y_pred = evaluate_model(model, X_train, y_train, X_test, y_test, name)
        results.append(metrics)
        predictions[name] = y_pred
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('test_rmse')
    
    print("\n" + "="*60)
    print("RESULTS SUMMARY")
    print("="*60)
    print(results_df.to_string(index=False))
    
    return results_df, predictions, models

print("Evaluation functions defined successfully!")

## 8. QB Model Training and Comparison

Treniranje svih modela za predviđanje Passing Yards kod Quarterback-a.

In [ ]:
# Compare all models for QB
qb_results, qb_predictions, qb_models = compare_all_models(
    X_train_qb, y_train_qb, 
    X_test_qb_scaled, y_test_qb, 
    "QB Passing Yards"
)

In [ ]:
# Visualize QB model comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# RMSE comparison
sorted_results = qb_results.sort_values('test_rmse')
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(sorted_results)))
axes[0].barh(sorted_results['model'], sorted_results['test_rmse'], color=colors)
axes[0].set_xlabel('Test RMSE (Yards)')
axes[0].set_title('QB Model Comparison: RMSE (lower is better)')
axes[0].invert_yaxis()

# MAE comparison
sorted_mae = qb_results.sort_values('test_mae')
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(sorted_mae)))
axes[1].barh(sorted_mae['model'], sorted_mae['test_mae'], color=colors)
axes[1].set_xlabel('Test MAE (Yards)')
axes[1].set_title('QB Model Comparison: MAE (lower is better)')
axes[1].invert_yaxis()

# R² comparison
sorted_r2 = qb_results.sort_values('test_r2', ascending=False)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(sorted_r2)))
axes[2].barh(sorted_r2['model'], sorted_r2['test_r2'], color=colors)
axes[2].set_xlabel('Test R²')
axes[2].set_title('QB Model Comparison: R² (higher is better)')
axes[2].invert_yaxis()

plt.suptitle('Quarterback (QB) Passing Yards Prediction - Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 9. RB Model Training and Comparison

Treniranje svih modela za predviđanje Rushing Yards kod Running Back-a.

In [ ]:
# Compare all models for RB
rb_results, rb_predictions, rb_models = compare_all_models(
    X_train_rb, y_train_rb, 
    X_test_rb_scaled, y_test_rb, 
    "RB Rushing Yards"
)

# Visualize RB model comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sorted_results = rb_results.sort_values('test_rmse')
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(sorted_results)))
axes[0].barh(sorted_results['model'], sorted_results['test_rmse'], color=colors)
axes[0].set_xlabel('Test RMSE (Yards)')
axes[0].set_title('RB Model Comparison: RMSE')
axes[0].invert_yaxis()

sorted_mae = rb_results.sort_values('test_mae')
axes[1].barh(sorted_mae['model'], sorted_mae['test_mae'], color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(sorted_mae))))
axes[1].set_xlabel('Test MAE (Yards)')
axes[1].set_title('RB Model Comparison: MAE')
axes[1].invert_yaxis()

sorted_r2 = rb_results.sort_values('test_r2', ascending=False)
axes[2].barh(sorted_r2['model'], sorted_r2['test_r2'], color=plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(sorted_r2))))
axes[2].set_xlabel('Test R²')
axes[2].set_title('RB Model Comparison: R²')
axes[2].invert_yaxis()

plt.suptitle('Running Back (RB) Rushing Yards Prediction - Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Random Forest with Hyperparameter Tuning

Random Forest kao primarni model sa optimizacijom hiperparametara korišćenjem GridSearchCV.

In [ ]:
# Hyperparameter tuning for Random Forest - QB Model
print("=" * 60)
print("RANDOM FOREST HYPERPARAMETER TUNING - QB")
print("=" * 60)

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)

# Grid search with cross-validation
print("Running GridSearchCV (this may take a few minutes)...")
grid_search = GridSearchCV(
    rf_model, 
    param_grid, 
    cv=5, 
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_qb, y_train_qb)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.2f}")

# Evaluate best model on test set
best_rf_qb = grid_search.best_estimator_
y_pred_qb_tuned = best_rf_qb.predict(X_test_qb_scaled)

print(f"\nTuned Random Forest - QB Test Performance:")
print(f"  MAE:  {mean_absolute_error(y_test_qb, y_pred_qb_tuned):.2f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test_qb, y_pred_qb_tuned)):.2f}")
print(f"  R²:   {r2_score(y_test_qb, y_pred_qb_tuned):.4f}")

## 11. Feature Importance Analysis

Analiza važnosti feature-a iz Random Forest modela - identifikacija ključnih prediktora performansi.

In [ ]:
# Extract feature importance from tuned QB model
qb_feature_importance = pd.DataFrame({
    'feature': X_train_qb.columns,
    'importance': best_rf_qb.feature_importances_
}).sort_values('importance', ascending=False)

print("QB Model - Top 15 Most Important Features for Passing Yards Prediction:")
print(qb_feature_importance.head(15).to_string(index=False))

# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 8))

top_features = qb_feature_importance.head(15)
bars = ax.barh(range(len(top_features)), top_features['importance'].values, color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance')
ax.set_title('QB Passing Yards - Random Forest Feature Importance\n(Similar to SHAP Analysis)')

# Add value labels
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width + 0.002, bar.get_y() + bar.get_height()/2, 
            f'{width:.3f}', ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Train and analyze RB Random Forest model
print("=" * 60)
print("RB RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 60)

# Train Random Forest on RB data
rf_rb = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_rb.fit(X_train_rb, y_train_rb)

# Extract feature importance
rb_feature_importance = pd.DataFrame({
    'feature': X_train_rb.columns,
    'importance': rf_rb.feature_importances_
}).sort_values('importance', ascending=False)

print("RB Model - Top 15 Most Important Features for Rushing Yards Prediction:")
print(rb_feature_importance.head(15).to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))

top_features = rb_feature_importance.head(15)
bars = ax.barh(range(len(top_features)), top_features['importance'].values, color='forestgreen')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance')
ax.set_title('RB Rushing Yards - Random Forest Feature Importance')

for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width + 0.002, bar.get_y() + bar.get_height()/2, 
            f'{width:.3f}', ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 12. Predictions vs Actual Values Visualization

Vizualizacija predikcija vs stvarnih vrednosti za evaluaciju kvaliteta modela.

In [ ]:
# Predictions vs Actual for QB
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# QB: Predicted vs Actual
y_pred_qb_rf = best_rf_qb.predict(X_test_qb_scaled)
r2_qb = r2_score(y_test_qb, y_pred_qb_rf)
rmse_qb = np.sqrt(mean_squared_error(y_test_qb, y_pred_qb_rf))

axes[0].scatter(y_test_qb, y_pred_qb_rf, alpha=0.6, edgecolors='none', s=50)
min_val = min(y_test_qb.min(), y_pred_qb_rf.min())
max_val = max(y_test_qb.max(), y_pred_qb_rf.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Passing Yards')
axes[0].set_ylabel('Predicted Passing Yards')
axes[0].set_title(f'QB: Predictions vs Actual\nR² = {r2_qb:.4f}, RMSE = {rmse_qb:.2f}')
axes[0].legend()

# RB: Predicted vs Actual
y_pred_rb_rf = rf_rb.predict(X_test_rb_scaled)
r2_rb = r2_score(y_test_rb, y_pred_rb_rf)
rmse_rb = np.sqrt(mean_squared_error(y_test_rb, y_pred_rb_rf))

axes[1].scatter(y_test_rb, y_pred_rb_rf, alpha=0.6, edgecolors='none', s=50, color='green')
min_val = min(y_test_rb.min(), y_pred_rb_rf.min())
max_val = max(y_test_rb.max(), y_pred_rb_rf.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Rushing Yards')
axes[1].set_ylabel('Predicted Rushing Yards')
axes[1].set_title(f'RB: Predictions vs Actual\nR² = {r2_rb:.4f}, RMSE = {rmse_rb:.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. Multi-Output Regression Extension

Proširenje: Istovremeno predviđanje više performansi igrača korišćenjem MultiOutputRegressor.

Na osnovu metodologije iz Elimam et al. (2025) - "Multi-Output Regression for the Prediction of World-Class Performances"

In [ ]:
# Multi-Output Regression for QB: Predict Yards, TD, and Passer Rating simultaneously
print("=" * 60)
print("MULTI-OUTPUT REGRESSION: QB")
print("Predicting: Passing Yards, TDs, and Passer Rating")
print("=" * 60)

# Define multiple targets for QB
qb_targets = ['Yds', 'TD', 'Rate']  # Passing Yards, Touchdowns, Passer Rating

# Prepare multi-target data
qb_multi_train = qb_train[qb_targets].dropna()
qb_multi_test = qb_test[qb_targets].dropna()

# Get common indices
train_idx = qb_multi_train.index.intersection(X_train_qb.index)
test_idx = qb_multi_test.index.intersection(X_test_qb_scaled.index)

X_train_multi = X_train_qb.loc[train_idx]
y_train_multi = qb_multi_train.loc[train_idx]
X_test_multi = X_test_qb_scaled.loc[test_idx]
y_test_multi = qb_multi_test.loc[test_idx]

print(f"Training samples: {len(X_train_multi)}")
print(f"Test samples: {len(X_test_multi)}")
print(f"Targets: {qb_targets}")

# Train Multi-Output Random Forest
mo_rf = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    n_jobs=-1
)

mo_rf.fit(X_train_multi, y_train_multi)
y_pred_multi = mo_rf.predict(X_test_multi)

# Evaluate each target
print("\nMulti-Output Regression Results:")
print("-" * 50)
for i, target in enumerate(qb_targets):
    mae = mean_absolute_error(y_test_multi.iloc[:, i], y_pred_multi[:, i])
    rmse = np.sqrt(mean_squared_error(y_test_multi.iloc[:, i], y_pred_multi[:, i]))
    r2 = r2_score(y_test_multi.iloc[:, i], y_pred_multi[:, i])
    print(f"{target:15s} MAE: {mae:8.2f}  RMSE: {rmse:8.2f}  R²: {r2:.4f}")

# Average RMSE (aRMSE) as per Elimam et al.
all_rmse = [np.sqrt(mean_squared_error(y_test_multi.iloc[:, i], y_pred_multi[:, i])) 
            for i in range(len(qb_targets))]
print("-" * 50)
print(f"Average RMSE (aRMSE): {np.mean(all_rmse):.2f}")

## 14. Final Results Summary and Conclusions

Sumiranje svih rezultata i poređenje performansi različitih pristupa.

In [ ]:
# Create comprehensive results summary
print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("NFL Player Performance Prediction")
print("=" * 70)

# Position comparison
position_results = {
    'QB (Passing Yards)': {
        'MAE': mean_absolute_error(y_test_qb, y_pred_qb_tuned),
        'RMSE': np.sqrt(mean_squared_error(y_test_qb, y_pred_qb_tuned)),
        'R²': r2_score(y_test_qb, y_pred_qb_tuned)
    },
    'RB (Rushing Yards)': {
        'MAE': mean_absolute_error(y_test_rb, y_pred_rb_rf),
        'RMSE': np.sqrt(mean_squared_error(y_test_rb, y_pred_rb_rf)),
        'R²': r2_score(y_test_rb, y_pred_rb_rf)
    }
}

# Create summary table
summary_data = []
for position, metrics in position_results.items():
    summary_data.append({
        'Position': position,
        'MAE': f"{metrics['MAE']:.2f}",
        'RMSE': f"{metrics['RMSE']:.2f}",
        'R²': f"{metrics['R²']:.4f}"
    })

summary_df = pd.DataFrame(summary_data)
print("\nBest Model Performance (Random Forest) by Position:")
print(summary_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

positions = list(position_results.keys())
mae_values = [position_results[p]['MAE'] for p in positions]
rmse_values = [position_results[p]['RMSE'] for p in positions]
r2_values = [position_results[p]['R²'] for p in positions]

# MAE
axes[0].bar(positions, mae_values, color=['steelblue', 'forestgreen'])
axes[0].set_ylabel('MAE (Yards)')
axes[0].set_title('Mean Absolute Error by Position')
for i, v in enumerate(mae_values):
    axes[0].text(i, v + 5, f'{v:.1f}', ha='center')

# RMSE
axes[1].bar(positions, rmse_values, color=['coral', 'coral'])
axes[1].set_ylabel('RMSE (Yards)')
axes[1].set_title('Root Mean Squared Error by Position')
for i, v in enumerate(rmse_values):
    axes[1].text(i, v + 5, f'{v:.1f}', ha='center')

# R²
axes[2].bar(positions, r2_values, color=['gold', 'gold'])
axes[2].set_ylabel('R² Score')
axes[2].set_title('R-Squared by Position')
axes[2].set_ylim(0, 1)
for i, v in enumerate(r2_values):
    axes[2].text(i, v + 0.02, f'{v:.3f}', ha='center')

plt.suptitle('NFL Performance Prediction - Position Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## Zaključci / Conclusions

### Ključni nalazi:

1. **Random Forest Regresija** se pokazala kao najefikasniji model za predviđanje performansi NFL igrača, što je u skladu sa literaturom (Frontiers in Sports and Active Living, 2025).

2. **Poziciono-specifično modelovanje** je ključno - svaka pozicija ima različite relevantne feature-e koji utiču na performanse.

3. **Feature Importance Analiza** je identifikovala najvažnije prediktore:
   - Za QB: Broj dodavanja (Att), procenat tačnih dodavanja (Cmp%), i metrike efikasnosti
   - Za RB: Broj pokušaja trčanja (Att), prosek jardi po pokušaju (Y/A), i jardi posle kontakta (YAC)

4. **Multi-Output Regresija** omogućava istovremeno predviđanje više performansi igrača uz održavanje dobre preciznosti.

### Praktična primena:
- NFL timovi mogu koristiti ovaj model za procenu vrednosti igrača prilikom formiranja tima
- Fantasy football industrija može poboljšati predikcije igrača

### Dalja proširenja:
- Implementacija neuronskih mreža
- Uključivanje dodatnih kontekstualnih faktora (protivnički tim, vremenski uslovi)
- Real-time predikcije tokom sezone

In [ ]:
# Save models for future use
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# Save the best QB model
joblib.dump(best_rf_qb, 'models/qb_random_forest_model.joblib')
print("QB model saved to: models/qb_random_forest_model.joblib")

# Save the RB model
joblib.dump(rf_rb, 'models/rb_random_forest_model.joblib')
print("RB model saved to: models/rb_random_forest_model.joblib")

# Save feature importance data
os.makedirs('results', exist_ok=True)
qb_feature_importance.to_csv('results/qb_feature_importance.csv', index=False)
rb_feature_importance.to_csv('results/rb_feature_importance.csv', index=False)

# Save model comparison results
qb_results.to_csv('results/qb_model_comparison.csv', index=False)
rb_results.to_csv('results/rb_model_comparison.csv', index=False)

print("\nAll results saved to: results/")
print("\n✅ Analysis Complete!")